# Process Mining & Time Series Pipeline

In [ ]:
# Import from library
import logging

# Import from external libraries
import pandas as pd

# Import from our package
from spi_time_series import PipelineBuilder
from spi_time_series.data import Dataset
from spi_time_series.data.schemas import EventLog, PreprocessedData, RawData
from spi_time_series.data.constants import VALID_END_ACTIVITIES
from spi_time_series.preprocessing.preprocess import (
    _build_trace_samples,
    clean_data,
    preprocess,
    sliding_window_factory,
    split_data,
)
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.features.extraction import extract_features_builder


# Set up logging
logging.basicConfig(level=logging.INFO, force=True)

## The Production Pipeline
Define functions required by the `PipelineBuilder` and executes the entire workflow from start to finish.


In [ ]:
def my_preprocessor(raw: RawData) -> EventLog:
    """Clean the raw event log."""

    cleaned = clean_data(raw, valid_ends=VALID_END_ACTIVITIES)
    return cleaned.event_log


def my_splitter(log: EventLog) -> PreprocessedData:
    """Split the cleaned log into train/test sets."""
    train_df, test_df = split_data(log)
    prefix_gen = sliding_window_factory(min_length=3)
    col_idx = {c: i for i, c in enumerate(train_df.columns)}

    return PreprocessedData(
        train_log=_build_trace_samples(train_df, prefix_gen),
        num_train_cases=len(train_df["case:concept:name"].unique()),
        test_log=_build_trace_samples(test_df, prefix_gen),
        num_test_cases=len(test_df["case:concept:name"].unique()),
        col_idx=col_idx,
    )


# Build the pipeline.
# As of now it only includes the dataset, preprocessor, and splitter, we will add feature extractors, models, and evaluators as needed.
dataset = Dataset()


def my_target_func(
    trace: pd.DataFrame, start_idx: int, end_idx: int
) -> int | str:
    return end_idx - start_idx


extractor = extract_features_builder(
    [BasicControlFlowFeatures()], my_target_func
)


pipeline = (
    PipelineBuilder()
    .with_dataset(dataset)
    .with_preprocessor(my_preprocessor)
    .with_splitter(my_splitter)
    .with_feature_extractor(
        extract_features_builder([BasicControlFlowFeatures()], my_target_func)
    )
    .build()
)
# Run the pipeline.
# evaluation = pipeline.run()
# Run the pipeline.
# evaluation = pipeline.run()

INFO:spi_time_series.data.dataset:Dataset found at /Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/data/raw/BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
/Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/.venv/lib/python3.14/site-packages/pm4py/utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


## Development & Debugging

Run these cells individually to dev and debug the outputs of each pipeline stage

## Data loading

In [3]:
# Quick check to see if the data is loaded correctly
dataset_dev = Dataset()
raw_dev = RawData(event_log=dataset_dev.log)

# Quick check to see if the data is loaded correctly
print(
    f"Loaded {len(raw_dev.event_log)} events with {raw_dev.event_log['case:concept:name'].nunique()} unique cases."
)


INFO:spi_time_series.data.dataset:Dataset found at /Users/sakkakis/Library/CloudStorage/OneDrive-StudentsRWTHAachenUniversity/Masters/Summer Senester 26/Supervised Process Intelligence/spi-time-series/data/raw/BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


Loaded 1202267 events with 31509 unique cases.


###  Preprocessing
Preprocessing does 3 steps:
1) Data Cleaning 
2) Data Splitting into train and test 
3) Prefix Generation + Target computation

#### Data Cleaning


In [ ]:
cleaned_dev = clean_data(raw_dev, valid_ends=VALID_END_ACTIVITIES)

# Check how many events and cases remain after cleaning
original_cases = raw_dev.event_log["case:concept:name"].nunique()
cleaned_cases = cleaned_dev.event_log["case:concept:name"].nunique()

print(f"Events remaining: {len(cleaned_dev.event_log):,}")
print(
    f"Cases remaining:  {cleaned_cases:,} (Retained {cleaned_cases / original_cases:.1%})"
)

INFO:spi_time_series.preprocessing.preprocess:Filtering for valid end activities: ['W_Validate application', 'W_Call after offers', 'W_Call incomplete files', 'O_Cancelled', 'A_Denied']
INFO:spi_time_series.preprocessing.preprocess:Remaining events after cleaning: 1193604


Events remaining: 1,193,604
Cases remaining:  31,232 (Retained 99.1%)


### Data Splitting


In [13]:
train_df, test_df = split_data(cleaned_dev.event_log)

# Check for leakage by comparing the max timestamp in the train set with the min timestamp in the test set
n_train = train_df["case:concept:name"].nunique()
n_test = test_df["case:concept:name"].nunique()
total_cases_after_split = n_train + n_test

# Calculate how many cases were dropped due to boundary overlap
original_cases = cleaned_dev.event_log["case:concept:name"].nunique()
dropped_cases = original_cases - total_cases_after_split

print(f"Train cases: {n_train:,} ({n_train / total_cases_after_split:.1%})")
print(f"Test cases:  {n_test:,} ({n_test / total_cases_after_split:.1%})")
print(f"Dropped cases (boundary overlap): {dropped_cases:,}\n")
train_end = train_df["time:timestamp"].max()
test_start = test_df["time:timestamp"].min()

if train_end <= test_start:
    print("✅ No leakage.")
else:
    print("❌ Leakage detected!")

INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.


Train cases: 22,723 (78.4%)
Test cases:  6,247 (21.6%)
Dropped cases (boundary overlap): 2,262

✅ No leakage.


### Prefix generation
Prefix are created and returned as generators.

In [14]:
raw = RawData(event_log=pipeline.dataset.log)
raw.event_log.head()

,lifecycle:transition,OfferedAmount,FirstWithdrawalAmount,NumberOfTerms,Selected,Accepted,OfferID,case:RequestedAmount,time:timestamp,MonthlyCost,EventID,EventOrigin,Action,case:ApplicationType,CreditScore,concept:name,org:resource,case:LoanGoal,case:concept:name
0,complete,NaN,NaN,NaN,None,None,None,20000.0,2016-01-01 09:51:15.304000+00:00,NaN,Application_652823628,Application,Created,New credit,NaN,A_Create Application,User_1,Existing loan takeover,Application_652823628
1,complete,NaN,NaN,NaN,None,None,None,20000.0,2016-01-01 09:51:15.352000+00:00,NaN,ApplState_1582051990,Application,statechange,New credit,NaN,A_Submitted,User_1,Existing loan takeover,Application_652823628
2,schedule,NaN,NaN,NaN,None,None,None,20000.0,2016-01-01 09:51:15.774000+00:00,NaN,Workitem_1298499574,Workflow,Created,New credit,NaN,W_Handle leads,User_1,Existing loan takeover,Application_652823628
3,withdraw,NaN,NaN,NaN,None,None,None,20000.0,2016-01-01 09:52:36.392000+00:00,NaN,Workitem_1673366067,Workflow,Deleted,New credit,NaN,W_Handle leads,User_1,Existing loan takeover,Application_652823628
4,schedule,NaN,NaN,NaN,None,None,None,20000.0,2016-01-01 09:52:36.403000+00:00,NaN,Workitem_1493664571,Workflow,Created,New credit,NaN,W_Complete application,User_1,Existing loan takeover,Application_652823628


In [15]:
cleaned = pipeline.preprocessor(raw)

INFO:spi_time_series.preprocessing.preprocess:Filtering for valid end activities: ['W_Validate application', 'W_Call after offers', 'W_Call incomplete files', 'O_Cancelled', 'A_Denied']
INFO:spi_time_series.preprocessing.preprocess:Remaining events after cleaning: 1193604


In [16]:
preprocessed = pipeline.splitter(cleaned)

INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.


In [17]:
features = pipeline.feature_extractor(preprocessed)

## Prefix generation
Prefix are created and returned as generators.

In [ ]:
from spi_time_series.preprocessing.preprocess import (
    preprocess,
    sliding_window_factory,
)

# define small dataframe for quick processing
dataset = Dataset()
small_cases = dataset.log["case:concept:name"].unique()[:5]
df_small = dataset.log[dataset.log["case:concept:name"].isin(small_cases)]

# raw = RawData(df_small)  # small datatset
raw = RawData(dataset.log)  # big dataset

INFO:spi_time_series.data.dataset:Dataset found at C:\Users\roman\Documents\Uni\Aachen\Semester_3\SoftwarePraktikum\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.


In [ ]:
prefix_generator = sliding_window_factory(min_length=3, max_length=None)

preprocessed_data = preprocess(
    raw,
    prefix_generator=prefix_generator,
    mode="log",
)

trace_sample = next(preprocessed_data.train_log)
start_idx, end_idx = next(trace_sample.prefix_indexes)

print(f"CASE ID: {trace_sample.case_id}")
print(
    pd.DataFrame(
        trace_sample.data[start_idx:end_idx],
        columns=list(preprocessed_data.col_idx.keys()),
    )
)

INFO:spi_time_series.preprocessing.preprocess:Remaining events after cleaning: 1202267
INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 22:39:59.933799936+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22895 cases, 
Test: 6302 cases.


CASE ID: Application_1000086665
  MonthlyCost OfferedAmount Selected          concept:name  \
0         NaN           NaN     None  A_Create Application   
1         NaN           NaN     None           A_Submitted   
2         NaN           NaN     None        W_Handle leads   

  case:ApplicationType  EventOrigin org:resource case:RequestedAmount  \
0           New credit  Application       User_1               5000.0   
1           New credit  Application       User_1               5000.0   
2           New credit     Workflow       User_1               5000.0   

  lifecycle:transition       Action  ...                   time:timestamp  \
0             complete      Created  ... 2016-08-03 15:57:21.673000+00:00   
1             complete  statechange  ... 2016-08-03 15:57:21.734000+00:00   
2             schedule      Created  ... 2016-08-03 15:57:21.963000+00:00   

                  EventID           case:LoanGoal OfferID NumberOfTerms  \
0  Application_1000086665  Other, see expl

## Feature Extraction

In [ ]:
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures


print(f"Number of train cases: {preprocessed_data.num_train_cases}")
feature_set = extract_features_builder(
    [BasicControlFlowFeatures()], my_target_func
)(preprocessed_data)

print(feature_set.X_train.head())

Number of train cases: 22895


Processing cases:  58%|█████▊    | 13295/22895 [02:15<01:19, 120.11it/s]